# Single-Route Training: Sydney-Hobart Routes

This notebook trains individual models for Sydney-Hobart and Hobart-Sydney routes to test if route-specific training improves performance compared to the multi-route model from notebook 6. The city pair was chosen due to the low performance identified in notebook 6. The cross-validation and K-fold techniques are also employed to identify potential train-test split problems.

**Motivation:**
From notebook 6, it was observed that route-specific features do capture route heterogeneity, but some routes are noticeably worse than others; particularly the Sydney - Hobart city pair:
- Hobart-Sydney, with R² = 0.1790 (worst overall)
- Sydney-Hobart, with R² = 0.3750

**Hypothesis:**
Training models on individual routes may improve performance by:
1. Eliminating noise from other routes
2. Allowing model to focus on route-specific patterns

**Model Configuration (from 4e):**
- `rainy_days_arr_exp` (regression) / `rainy_days_arr` (classification)
- `delay_rate_gradient` (momentum)
- `temp_volatility_total_exp`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    accuracy_score, f1_score, roc_auc_score
)
import warnings
warnings.filterwarnings('ignore')

# Plotting style
plt.rcParams['text.usetex'] = False
plt.rcParams['font.family'] = 'serif'
plt.rcParams['axes.linewidth'] = 1.5

# XGBoost for classification
try:
    import xgboost as xgb
    HAS_XGB = True
    print("XGBoost available")
except ImportError:
    HAS_XGB = False
    print("XGBoost not installed (pip install xgboost)")

%matplotlib inline

## Reference Values

Multi-route model performance from notebook 6 (Ridge, by route):

In [ ]:
# Reference values from notebook 6 (multi-route model, performance by route)
ref_multiroute = {
    'Sydney_Hobart': {'R2': 0.3750, 'RMSE': 0.0968, 'MAE': 0.0697},
    'Hobart_Sydney': {'R2': 0.1790, 'RMSE': 0.1125, 'MAE': 0.0684}
}

# Reference values from notebook 4e (single-route SYD-MEL)
ref_4e = {
    'Ridge': {'R2': 0.4941, 'RMSE': 0.0703},
    'Random Forest': {'R2': 0.4715, 'RMSE': 0.0719},
    'Logistic': {'F1': 0.7746, 'AUC': 0.8852},
    'Random Forest_clf': {'F1': 0.7514, 'AUC': 0.8555},
    'XGBoost': {'F1': 0.8065, 'AUC': 0.8742}
}

print("Reference: Multi-route model performance (by route)")
print("-" * 50)
for route, metrics in ref_multiroute.items():
    print(f"  {route}: R²={metrics['R2']:.4f}, RMSE={metrics['RMSE']:.4f}")

## 1. Data Preparation

In [ ]:
# Load multi-route data
df = pd.read_csv('../data/processed/ml_training_data_multiroute.csv')

# Parse dates
df['year_month_dt'] = pd.to_datetime(df['year_month'])
df['month_num'] = df['year_month_dt'].dt.month
df['year'] = df['year'].astype(int)

# Create route identifier
df['route'] = df['departing_port'] + '_' + df['arriving_port']

# Filter to Sydney-Hobart routes only
target_routes = ['Sydney_Hobart', 'Hobart_Sydney']
df_routes = df[df['route'].isin(target_routes)].copy()

print(f"Full dataset: {len(df)} records")
print(f"Filtered to target routes: {len(df_routes)} records")
print(f"\nRoute distribution:")
print(df_routes['route'].value_counts())

In [ ]:
# Create unique identifier for each airline-route combination
df_routes['airline_route'] = df_routes['airline'] + '_' + df_routes['route']

# Sort for proper lag creation
df_routes = df_routes.sort_values(['airline_route', 'year_month_dt']).reset_index(drop=True)

# Create lagged features
df_routes['delay_rate_lag1'] = df_routes.groupby('airline_route')['delay_rate'].shift(1)
df_routes['delay_rate_lag2'] = df_routes.groupby('airline_route')['delay_rate'].shift(2)

# Create momentum feature (from 4d)
df_routes['delay_rate_gradient'] = df_routes['delay_rate_lag1'] - df_routes['delay_rate_lag2']

# Create cyclical month encoding
df_routes['month_sin'] = np.sin(2 * np.pi * df_routes['month_num'] / 12)
df_routes['month_cos'] = np.cos(2 * np.pi * df_routes['month_num'] / 12)

# One-hot encode airline (for within-route variation)
airline_dummies = pd.get_dummies(df_routes['airline'], prefix='airline')
df_routes = pd.concat([df_routes, airline_dummies], axis=1)
airline_cols = list(airline_dummies.columns)

# Create exponential transformations (from 4c, 4e)
df_routes['rainy_days_arr_exp'] = np.exp(df_routes['rainy_days_arr'] / df_routes['rainy_days_arr'].max())
df_routes['temp_volatility_total'] = df_routes['temp_volatility_dep'] + df_routes['temp_volatility_arr']
df_routes['temp_volatility_total_exp'] = np.exp(df_routes['temp_volatility_total'] / df_routes['temp_volatility_total'].max())

print(f"Airlines in dataset: {df_routes['airline'].nunique()}")
print(f"Airline columns: {airline_cols}")

# Drop rows with missing lag values
df_clean = df_routes.dropna(subset=['delay_rate_lag1', 'delay_rate_lag2', 'delay_rate_gradient']).copy()
print(f"\nRows after dropping NaN lags: {len(df_clean)}")

## 2. Cross-Validation vs Train/Test Split

### Prior Approach (Train/Val/Test Split):
```
|-------- Train --------|-- Val --|-- Test --|
   2010-2017, 2024        2018,2023   2019,2025
```
- **Single evaluation**: Model is trained once, evaluated once on test set
- **Problem**: With small datasets (~350 samples per route), the single test set (77 samples) may not be representative
- **Risk**: Results can be lucky or unlucky depending on which samples end up in test

### Cross-Validation:
```
Fold 1: |---Train---|--Test--|---Train---|
Fold 2: |---Train---------|--Test--|--Train--|
Fold 3: |---Train---------------|--Test--|
...
```
- **Multiple evaluations**: Data is split into K folds, each fold takes turns being the test set
- **Benefit**: Every sample gets tested exactly once, giving more robust performance estimate
- **Result**: Reports mean ± std of metrics across all folds

### Time-Series Cross-Validation:
Regular K-fold randomly shuffles data, which breaks temporal order. For time-series:
```
Fold 1: |--Train--|--Test--|
Fold 2: |----Train----|--Test--|
Fold 3: |------Train------|--Test--|
```
- Training always uses past data to predict future (respects causality)
- Each fold expands the training window

### Why This Helps With Small Datasets:
- More reliable performance estimate (not dependent on one lucky/unlucky split)
- Uses all data for both training and testing (just not simultaneously)
- Reveals if model is consistently good or inconsistent across time periods

In [ ]:
# Define feature sets (matching 4e configuration, without route dummies)
base_features = airline_cols + ['month_sin', 'month_cos', 'delay_rate_lag1', 'sectors_scheduled']

# Regression: rainy_days_arr_exp + gradient + temp_volatility_total_exp
reg_features = base_features + ['rainy_days_arr_exp', 'delay_rate_gradient', 'temp_volatility_total_exp']

# Classification: rainy_days_arr (plain) + gradient + temp_volatility_total_exp
clf_features = base_features + ['rainy_days_arr', 'delay_rate_gradient', 'temp_volatility_total_exp']

print(f"Base features: {len(base_features)}")
print(f"  - Airline dummies: {len(airline_cols)}")
print(f"  - Other: month_sin, month_cos, delay_rate_lag1, sectors_scheduled")
print(f"\nRegression features: {len(reg_features)}")
print(f"Classification features: {len(clf_features)}")
print(f"\nNote: No route dummies needed (training per-route)")

## 3. Time-Series Cross-Validation Implementation

We'll use `TimeSeriesSplit` from sklearn, which respects temporal ordering.

In [ ]:
from sklearn.model_selection import TimeSeriesSplit

def train_with_cv(df_route, route_name, reg_features, clf_features, n_splits=5):
    """
    Train models using Time-Series Cross-Validation.
    Returns dict of results with mean and std across folds.
    """
    # Exclude COVID period (2020-2022) for cleaner evaluation
    df_route = df_route[~df_route['year'].isin([2020, 2021, 2022])].copy()
    
    # Sort by time for proper time-series split
    df_route = df_route.sort_values('year_month_dt').reset_index(drop=True)
    
    print(f"\n{'='*70}")
    print(f"ROUTE: {route_name} (Time-Series Cross-Validation, {n_splits} folds)")
    print(f"{'='*70}")
    print(f"Total samples (excluding COVID): {len(df_route)}")
    
    # Prepare features
    X_reg = df_route[reg_features].values
    X_clf = df_route[clf_features].values
    y_reg = df_route['delay_rate'].values
    y_clf = df_route['is_high_delay'].values
    
    # Time-series split
    tscv = TimeSeriesSplit(n_splits=n_splits)
    
    # Storage for fold results
    ridge_r2_scores = []
    rf_r2_scores = []
    logreg_f1_scores = []
    rf_clf_f1_scores = []
    xgb_f1_scores = []
    
    fold_details = []
    
    for fold, (train_idx, test_idx) in enumerate(tscv.split(X_reg)):
        # Get train/test years for this fold
        train_years = df_route.iloc[train_idx]['year'].unique()
        test_years = df_route.iloc[test_idx]['year'].unique()
        
        # Regression
        X_train, X_test = X_reg[train_idx], X_reg[test_idx]
        y_train, y_test = y_reg[train_idx], y_reg[test_idx]
        
        # Scale
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)
        
        # Ridge
        ridge = Ridge(alpha=1.0)
        ridge.fit(X_train_scaled, y_train)
        ridge_pred = ridge.predict(X_test_scaled)
        ridge_r2 = r2_score(y_test, ridge_pred)
        ridge_r2_scores.append(ridge_r2)
        
        # Random Forest Regression
        rf_reg = RandomForestRegressor(n_estimators=100, max_depth=10, min_samples_leaf=5, random_state=42, n_jobs=-1)
        rf_reg.fit(X_train, y_train)
        rf_pred = rf_reg.predict(X_test)
        rf_r2 = r2_score(y_test, rf_pred)
        rf_r2_scores.append(rf_r2)
        
        # Classification
        X_train_clf, X_test_clf = X_clf[train_idx], X_clf[test_idx]
        y_train_clf, y_test_clf = y_clf[train_idx], y_clf[test_idx]
        
        scaler_clf = StandardScaler()
        X_train_clf_scaled = scaler_clf.fit_transform(X_train_clf)
        X_test_clf_scaled = scaler_clf.transform(X_test_clf)
        
        # Logistic Regression
        logreg = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
        logreg.fit(X_train_clf_scaled, y_train_clf)
        logreg_pred = logreg.predict(X_test_clf_scaled)
        logreg_f1 = f1_score(y_test_clf, logreg_pred, zero_division=0)
        logreg_f1_scores.append(logreg_f1)
        
        # Random Forest Classifier
        rf_clf = RandomForestClassifier(n_estimators=100, max_depth=10, min_samples_leaf=5, random_state=42, n_jobs=-1)
        rf_clf.fit(X_train_clf, y_train_clf)
        rf_clf_pred = rf_clf.predict(X_test_clf)
        rf_clf_f1 = f1_score(y_test_clf, rf_clf_pred, zero_division=0)
        rf_clf_f1_scores.append(rf_clf_f1)
        
        # XGBoost
        if HAS_XGB:
            xgb_clf = xgb.XGBClassifier(
                n_estimators=100, max_depth=5, learning_rate=0.1,
                min_child_weight=5, random_state=42, n_jobs=-1, verbosity=0
            )
            xgb_clf.fit(X_train_clf, y_train_clf)
            xgb_pred = xgb_clf.predict(X_test_clf)
            xgb_f1 = f1_score(y_test_clf, xgb_pred, zero_division=0)
            xgb_f1_scores.append(xgb_f1)
        
        fold_details.append({
            'fold': fold + 1,
            'train_size': len(train_idx),
            'test_size': len(test_idx),
            'train_years': f"{min(train_years)}-{max(train_years)}",
            'test_years': f"{min(test_years)}-{max(test_years)}",
            'ridge_r2': ridge_r2,
            'rf_r2': rf_r2,
            'logreg_f1': logreg_f1
        })
    
    # Print fold details
    print(f"\n{'Fold':<6} {'Train':>8} {'Test':>6} {'Train Years':<12} {'Test Years':<12} {'Ridge R²':>10} {'RF R²':>10}")
    print("-" * 70)
    for fd in fold_details:
        print(f"{fd['fold']:<6} {fd['train_size']:>8} {fd['test_size']:>6} {fd['train_years']:<12} {fd['test_years']:<12} {fd['ridge_r2']:>10.4f} {fd['rf_r2']:>10.4f}")
    
    # Calculate summary statistics
    results = {
        'route': route_name,
        'n_samples': len(df_route),
        'regression': {
            'Ridge': {
                'R2_mean': np.mean(ridge_r2_scores),
                'R2_std': np.std(ridge_r2_scores),
                'R2_scores': ridge_r2_scores
            },
            'Random Forest': {
                'R2_mean': np.mean(rf_r2_scores),
                'R2_std': np.std(rf_r2_scores),
                'R2_scores': rf_r2_scores
            }
        },
        'classification': {
            'Logistic': {
                'F1_mean': np.mean(logreg_f1_scores),
                'F1_std': np.std(logreg_f1_scores)
            },
            'Random Forest': {
                'F1_mean': np.mean(rf_clf_f1_scores),
                'F1_std': np.std(rf_clf_f1_scores)
            }
        },
        'fold_details': fold_details
    }
    
    if HAS_XGB:
        results['classification']['XGBoost'] = {
            'F1_mean': np.mean(xgb_f1_scores),
            'F1_std': np.std(xgb_f1_scores)
        }
    
    # Print summary
    print(f"\n{'Model':<20} {'Mean R²':>12} {'Std':>10}")
    print("-" * 45)
    print(f"{'Ridge':<20} {results['regression']['Ridge']['R2_mean']:>12.4f} {results['regression']['Ridge']['R2_std']:>10.4f}")
    print(f"{'Random Forest':<20} {results['regression']['Random Forest']['R2_mean']:>12.4f} {results['regression']['Random Forest']['R2_std']:>10.4f}")
    
    print(f"\n{'Model':<20} {'Mean F1':>12} {'Std':>10}")
    print("-" * 45)
    for model in results['classification']:
        print(f"{model:<20} {results['classification'][model]['F1_mean']:>12.4f} {results['classification'][model]['F1_std']:>10.4f}")
    
    return results

In [ ]:
# Run time-series cross-validation for each route
cv_results = {}

for route in target_routes:
    df_route = df_clean[df_clean['route'] == route].copy()
    results = train_with_cv(df_route, route, reg_features, clf_features, n_splits=5)
    cv_results[route] = results

## 4. Cross-Validation Results vs Multi-Route Model

In [ ]:
print("="*80)
print("COMPARISON: Single-Route CV vs Multi-Route Model (Notebook 6)")
print("="*80)

print("\nREGRESSION (Ridge R²):")
print("-" * 80)
print(f"{'Route':<20} {'Multi-Route':>12} {'Single CV Mean':>14} {'± Std':>10} {'Δ':>10} {'Verdict':>12}")
print("-" * 80)

cv_comparison = []
for route in target_routes:
    multi_r2 = ref_multiroute[route]['R2']
    cv_mean = cv_results[route]['regression']['Ridge']['R2_mean']
    cv_std = cv_results[route]['regression']['Ridge']['R2_std']
    diff = cv_mean - multi_r2
    sign = '+' if diff > 0 else ''
    
    # Consider std in verdict
    if diff > cv_std:
        verdict = "IMPROVED"
    elif diff < -cv_std:
        verdict = "WORSE"
    else:
        verdict = "SIMILAR"
    
    print(f"{route:<20} {multi_r2:>12.4f} {cv_mean:>14.4f} {cv_std:>10.4f} {sign}{diff:>9.4f} {verdict:>12}")
    cv_comparison.append({
        'Route': route,
        'Multi_R2': multi_r2,
        'CV_Mean': cv_mean,
        'CV_Std': cv_std,
        'Diff': diff
    })

print("\nNote: Verdict considers whether improvement exceeds 1 standard deviation")

## 5. Fold-by-Fold Analysis

The variance across folds is examined below to assess whether performance is consistent or varies by time period.

In [ ]:
# Fold-by-fold R² visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
palette = sns.color_palette("Set1", 2)

for idx, route in enumerate(target_routes):
    ax = axes[idx]
    fold_details = cv_results[route]['fold_details']
    
    folds = [fd['fold'] for fd in fold_details]
    ridge_r2s = [fd['ridge_r2'] for fd in fold_details]
    rf_r2s = [fd['rf_r2'] for fd in fold_details]
    
    x = np.arange(len(folds))
    width = 0.35
    
    ax.bar(x - width/2, ridge_r2s, width, label='Ridge', color=palette[0], alpha=0.8)
    ax.bar(x + width/2, rf_r2s, width, label='Random Forest', color=palette[1], alpha=0.8)
    
    # Reference line
    ax.axhline(ref_multiroute[route]['R2'], color='green', linestyle='--', 
               linewidth=1.5, label=f'Multi-Route: {ref_multiroute[route]["R2"]:.3f}')
    
    ax.set_xlabel('Fold')
    ax.set_ylabel(r'$R^2$')
    ax.set_title(f'{route.replace("_", "-")}: R² by CV Fold')
    ax.set_xticks(x)
    ax.set_xticklabels([f'Fold {f}' for f in folds])
    ax.legend(loc='upper left', fontsize=9)
    ax.set_ylim(-0.5, 0.7)  # Allow negative R² to show poor folds

plt.tight_layout()
plt.show()

# Print insights
print("\nKey Observations:")
for route in target_routes:
    fold_details = cv_results[route]['fold_details']
    r2_scores = [fd['ridge_r2'] for fd in fold_details]
    min_r2 = min(r2_scores)
    max_r2 = max(r2_scores)
    print(f"\n{route}:")
    print(f"  R² range: {min_r2:.4f} to {max_r2:.4f} (spread: {max_r2-min_r2:.4f})")
    print(f"  High variance indicates inconsistent performance across time periods")

In [ ]:
# Visualization: CV results with error bars
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Ridge R² comparison with error bars
ax = axes[0]
x = np.arange(len(target_routes))
width = 0.35

multi_r2_vals = [ref_multiroute[r]['R2'] for r in target_routes]
cv_means = [cv_results[r]['regression']['Ridge']['R2_mean'] for r in target_routes]
cv_stds = [cv_results[r]['regression']['Ridge']['R2_std'] for r in target_routes]

bars1 = ax.bar(x - width/2, multi_r2_vals, width, label='Multi-Route (6)', color=palette[0], alpha=0.8)
bars2 = ax.bar(x + width/2, cv_means, width, yerr=cv_stds, capsize=5, 
               label='Single-Route CV (6a)', color=palette[1], alpha=0.8)

ax.axhline(ref_4e['Ridge']['R2'], color='green', linestyle='--', linewidth=1.5, 
           label=f'4e SYD-MEL: R²={ref_4e["Ridge"]["R2"]:.4f}')
ax.set_ylabel(r'Test $R^2$')
ax.set_title(r'Ridge Regression: $R^2$ with CV Error Bars')
ax.set_xticks(x)
ax.set_xticklabels([r.replace('_', '-') for r in target_routes])
ax.legend(loc='upper right', fontsize=9)
ax.set_ylim(-0.2, 0.7)

# Plot 2: R² distribution across folds (box plot style)
ax = axes[1]

# Prepare data for box plot
ridge_scores_by_route = [cv_results[r]['regression']['Ridge']['R2_scores'] for r in target_routes]
rf_scores_by_route = [cv_results[r]['regression']['Random Forest']['R2_scores'] for r in target_routes]

positions_ridge = np.arange(len(target_routes)) * 2
positions_rf = positions_ridge + 0.6

bp1 = ax.boxplot(ridge_scores_by_route, positions=positions_ridge, widths=0.5, 
                  patch_artist=True, boxprops=dict(facecolor=palette[0], alpha=0.7))
bp2 = ax.boxplot(rf_scores_by_route, positions=positions_rf, widths=0.5,
                  patch_artist=True, boxprops=dict(facecolor=palette[1], alpha=0.7))

# Add reference lines for multi-route
for i, route in enumerate(target_routes):
    ax.hlines(ref_multiroute[route]['R2'], positions_ridge[i]-0.4, positions_rf[i]+0.4,
              color='green', linestyle='--', linewidth=2)

ax.set_xticks(positions_ridge + 0.3)
ax.set_xticklabels([r.replace('_', '-') for r in target_routes])
ax.set_ylabel(r'$R^2$ across folds')
ax.set_title('R² Distribution Across CV Folds\n(blue=Ridge, red=RF, green=Multi-Route ref)')
ax.legend([bp1["boxes"][0], bp2["boxes"][0]], ['Ridge', 'Random Forest'], loc='upper right')

plt.tight_layout()
plt.show()

## 6. Summary and Observations

In [ ]:
print("="*80)
print("SUMMARY: Single-Route Training with Cross-Validation")
print("="*80)

print("\nDataset Info:")
print("-" * 60)
for route in target_routes:
    n_samples = cv_results[route]['n_samples']
    print(f"  {route}: {n_samples} samples (excluding COVID)")

print("\n" + "="*80)
print("CROSS-VALIDATION RESULTS (Ridge R²)")
print("="*80)
print(f"\n{'Route':<20} {'Multi-Route':>12} {'CV Mean':>12} {'CV Std':>10} {'Δ':>10} {'Verdict':>12}")
print("-" * 80)

for route in target_routes:
    multi_r2 = ref_multiroute[route]['R2']
    cv_mean = cv_results[route]['regression']['Ridge']['R2_mean']
    cv_std = cv_results[route]['regression']['Ridge']['R2_std']
    diff = cv_mean - multi_r2
    sign = '+' if diff > 0 else ''
    
    if diff > cv_std:
        verdict = "IMPROVED"
    elif diff < -cv_std:
        verdict = "WORSE"
    else:
        verdict = "SIMILAR"
    
    print(f"{route:<20} {multi_r2:>12.4f} {cv_mean:>12.4f} {cv_std:>10.4f} {sign}{diff:>9.4f} {verdict:>12}")

print("\n" + "="*80)
print("KEY FINDINGS")
print("="*80)

# Calculate metrics
avg_cv_mean = np.mean([cv_results[r]['regression']['Ridge']['R2_mean'] for r in target_routes])
avg_multi = np.mean([ref_multiroute[r]['R2'] for r in target_routes])
avg_diff = avg_cv_mean - avg_multi

print(f"\n1. Average single-route CV R²: {avg_cv_mean:.4f}")
print(f"2. Average multi-route R²: {avg_multi:.4f}")
print(f"3. Average difference: {'+' if avg_diff > 0 else ''}{avg_diff:.4f}")

# Variance analysis
for route in target_routes:
    cv_std = cv_results[route]['regression']['Ridge']['R2_std']
    r2_scores = cv_results[route]['regression']['Ridge']['R2_scores']
    print(f"\n> {route} CV variance:")
    print(f"   - Std: {cv_std:.4f}")
    print(f"   - Range: {min(r2_scores):.4f} to {max(r2_scores):.4f}")
    if cv_std > 0.15:
        print(f"   - HIGH VARIANCE: Performance is inconsistent across time periods")


### Observations

High variance was detected in Sydney_Hobart route's raw data:
- This suggests the route is inherently difficult to predict
- Performance varies significantly depending on the time period

These observations suggest that neither single-route nor multi-route will perform well. If anything, the notebook shows that single-route training performs **worse**. Probably because multi-route training benefits from a larger training set.

Thus, the best guess so far is that Hobart routes are harder to predict probably due to:
- its smaller sample sizes per route (data limitation)
- its nature being a more regional airport (naturally higher delay variability)

## 7. Next Step

However, Hobart–Melbourne performs well, which suggests the issue is not the nature of the Hobart airport per se. The hypothesis that smaller sample size is the cause will be explored in the next notebook.